# 🇮🇶 Gas Turbine Thermal Derating in Extreme Summer Heat (>50°C) — Basra, Iraq
### Modeling Compressor Air Density Loss and Evaporative Fogging Power Recovery in Shatt Al-Basra / Rumaila
**Author:** Mohammed Basra (`mohammedbasra-star`)  

In Southern Iraq (Basra Governorate), summer temperatures frequently exceed $50^\circ\text{C}$ ($122^\circ\text{F}$).
This tutorial demonstrates:
1. **Compressor Air Density Derating:** Modeling mass flow degradation from ISO $15^\circ\text{C}$ to $52^\circ\text{C}$.
2. **Psychrometric Fogging Limits:** Using the Stull formula to evaluate evaporative inlet air chilling potential.
3. **Physics-Guided Machine Learning (PGNN):** Predicting power output while enforcing $\frac{\partial P}{\partial T_{amb}} \le 0$.

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))
from thermodynamics.brayton_derating import simulate_turbine_power, compute_air_density
from thermodynamics.psychrometric_cooling import evaluate_fogging_potential

## 1. Thermodynamic Derating Profile across Basra Temperature Range (10°C to 52°C)

In [ ]:
temps = np.linspace(10, 52, 50)
powers = [simulate_turbine_power(t, P_mbar=1010.0, RH_percent=25.0)['power_mw'] for t in temps]
densities = [compute_air_density(t, P_mbar=1010.0, RH_percent=25.0) for t in temps]

fig, ax1 = plt.subplots(figsize=(8, 4.5))
color = 'tab:red'
ax1.set_xlabel('Basra Ambient Temperature (°C)')
ax1.set_ylabel('Net Power Output (MW)', color=color)
ax1.plot(temps, powers, color=color, lw=2.5)
ax1.tick_params(axis='y', labelcolor=color)
ax1.axvline(50.0, color='darkred', linestyle='--', label='50°C Summer Peak')

ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Compressor Air Density (kg/m³)', color=color)
ax2.plot(temps, densities, color=color, lw=2, linestyle=':')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Gas Turbine Ambient Derating Under Extreme Basra Climatic Conditions')
fig.tight_layout()
plt.show()

## 2. Evaporative Fogging Power Recovery Potential

In [ ]:
t_amb = 50.0
rh_values = np.linspace(10, 70, 30)
power_gains = []
for rh in rh_values:
    fog = evaluate_fogging_potential(t_amb, rh)
    p_cooled = simulate_turbine_power(fog['cooled_inlet_temp_c'], 1010.0, 90.0)['power_mw']
    p_uncooled = simulate_turbine_power(t_amb, 1010.0, rh)['power_mw']
    power_gains.append(p_cooled - p_uncooled)

plt.figure(figsize=(7, 4))
plt.plot(rh_values, power_gains, 'b-o', lw=2)
plt.xlabel('Relative Humidity (%) during 50°C Summer Day')
plt.ylabel('Recoverable Power Capacity (MW)')
plt.title('Inlet Fogging Power Recovery vs Basra Ambient Humidity')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()